In [64]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА ДАННЫХ
# Загружаем предварительно очищенные данные по прибывшим и выбывшим
df_arrivals = pd.read_csv('data/clean_count_of_arrivals.csv', sep=';', encoding='utf-8-sig')
df_dropped_out = pd.read_csv('data/clean_count_of_dropped_out.csv', sep=';', encoding='utf-8-sig')

# Извлекаем список лет (колонки с данными начинаются с 3-го индекса)
cols = df_arrivals.columns[2:].tolist()

# 2. ОБЪЕДИНЕНИЕ И РАСЧЕТ САЛЬДО
# Объединяем таблицы по региону и коду для сопоставления потоков
df_merged = pd.merge(df_arrivals, df_dropped_out, on=['Регион', 'Код'], suffixes=('_in', '_out'))

# Рассчитываем чистый миграционный прирост (сальдо) для каждого года
for year in cols:
    df_merged[f'{year}'] = df_merged[f'{year}_in'] - df_merged[f'{year}_out']

# Оставляем только итоговые расчетные колонки
result_cols = ['Регион', 'Код'] + [f'{year}' for year in cols]
df_final = df_merged[result_cols]

# 3. ЭКСПОРТ
# Сохранение финального отчета для последующей визуализации
df_final.to_csv('data/test.csv', index=False, sep=';', encoding='utf-8-sig')

In [71]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА ДАННЫХ
# Загружаем предварительно очищенные данные по общему количеству жителей и миграционному приросту
df_ammount = pd.read_csv('data/input_clean/clean_count_of_arrivals.csv', sep=';', encoding='utf-8-sig')
df_migrations = pd.read_csv('data/input_clean/clean_migration_increase.csv', sep=';', encoding='utf-8-sig')

# Извлекаем список лет (колонки с данными начинаются с 3-го индекса)
cols = df_ammount.columns[2:].tolist()

# 2. ОБЪЕДИНЕНИЕ И РАСЧЕТ САЛЬДО
# Объединяем таблицы по региону и коду для сопоставления потоков
df_merged = pd.merge(df_ammount, df_migrations, on=['Регион', 'Код'], suffixes=('_ammount', '_migr'))

# Рассчитываем коэффициент миграционного прироста для каждого года
for year in cols:
    df_merged[f'{year}'] =  (df_merged[f'{year}_migr'] / df_merged[f'{year}_ammount'] * 1000).round(2)

# Оставляем только итоговые расчетные колонки
result_cols = ['Регион', 'Код'] + [f'{year}' for year in cols]
df_final = df_merged[result_cols]
df_final = df_final.drop(0).reset_index(drop=True)

# 3. ЭКСПОРТ
# Сохранение финального отчета для последующей визуализации
df_final.to_csv('data/reports/test.csv', index=False, sep=';', encoding='utf-8-sig')

In [151]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА ДАННЫХ
# Загружаем данные с коэффициентом миграционного прироста
df = pd.read_csv('data/reports/growth_rate.csv', sep=';', encoding='utf-8-sig')

# Извлекаем список лет (колонки с данными начинаются с 3-го индекса)
cols = df.columns[2:].tolist()

# 2. РАССЧЁТ ТРЕНДОВ
# Создаём столбцы для линейного и недавнего трендов, а затем расчитываем их
df_summary = pd.DataFrame()
df_summary['Линейный тренд'] = None
df_summary['Недавний тренд'] = None
df_summary['Линейный тренд'] = (df[cols[-1]] - df[cols[0]]).round(2)
df_summary['Недавний тренд'] = (df[cols[-1]] - df[cols[-2]]).round(2)

# 3. ОПРЕДЕЛЕНИЕ ПРИВЛЕКТЕЛЬНОСТИ
# Определяем привлекательность региона по коэффициенту миграционного прироста в каждом году
def is_attractive(num):
    return 1 if num > 0 else 0
for col in cols:
    df[col] = df[col].apply(is_attractive)
    
# 4. ЭКСПОРТ
# Сохранение финального отчета для последующей визуализации
df.to_csv('data/reports/attractiveness_by_years.csv', index=False, sep=';', encoding='utf-8-sig')

# 5. ПРОДОЛЖАЕМ ЗАПОЛНЕНИЕ ИТОГОВОЙ ТАБЛИЦЫ
# Создаём оставшиеся столбцы 
df_summary['Регион'] = df['Регион']
df_summary['Код'] = df['Код']
df_summary['% привлекательности'] = None
df_summary['Категория'] = None
df_summary['Вектор тренда'] = None

# Рассчёт % привлекательности за все года
df_summary['% привлекательности'] = (df[cols].sum(axis=1)/len(cols)*100).round(1)

# На основе % привелекательности определяем категорию региона
categoryes = ['Магниты', 'Нестабильные', 'Доноры']

def define_category(val):
    if val > 70: return categoryes[0]
    elif val >= 30 and val <= 70: return categoryes[1] 
    else: return categoryes[2]

df_summary['Категория'] = df_summary['% привлекательности'].apply(define_category)

df_summary['% привлекательности'] = (df[cols].sum(axis=1)/len(cols)*100).round(1)


# Определяем вектор тренда на основе недавнего тренда
def get_trend_vector(val):
    if val > 1.0: return "📈 Сильный рост"
    elif val > 0.1: return "↗️ Улучшение"
    elif val < -1.0: return "📉 Сильный спад"
    elif val < -0.1: return "↘️ Ухудшение"
    else: return "➡️ Стабильно"

df_summary['Вектор тренда'] = df_summary['Недавний тренд'].apply(get_trend_vector)

# 6. ФИНАЛЬНЙ ЭКСПОРТ
df_summary = df_summary[selected_columns].copy()
df_summary.to_csv('data/reports/attractiveness_summary.csv', index=False, sep=';', encoding='utf-8-sig')

In [141]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# Загружаем очищенные данные по миграционному приросту в разрезе пола и местности
df = pd.read_csv('data/input_clean/clean_migration_increase_per_gender.csv', sep=';', encoding='utf-8-sig')

# Определяем список колонок с годами (начиная с 5-й колонки)
cols = df.columns[4:].tolist()

# Агрегируем данные: суммируем показатели пола, сохраняя разделение по Региону, Коду и Местности
df = df.groupby(['Регион', 'Код', 'Местность'])[cols].sum().reset_index()

df_summary = df.copy()
df_summary['Динамика потока'] = None

# 2. РАСЧЕТ МИГРАЦИОННОЙ ДИНАМИКИ
# Считаем абсолютное изменение миграционного потока между последним и первым доступными годами
# Это показывает, стал ли регион более или менее привлекательным за весь период
df_summary['Динамика потока'] = (df_summary[cols[-1]] - df_summary[cols[0]]).round(2)

# 3. ПЕРЕСТРОЙКА СТРУКТУРЫ ТАБЛИЦЫ (PIVOT)
# Переводим типы местности (Город/Село) из строк в отдельные колонки для сопоставления внутри региона
df_summary = df_summary.pivot(index=['Регион', 'Код'], 
                            columns='Местность', 
                            values='Динамика потока').reset_index()
# Переименовываем колонки для наглядности аналитического отчета
df_summary = df_summary.rename(columns={
    'Городская местность': 'Городская динамика',
    'Сельская местность': 'Сельская динамика'
})
# Заполняем пропуски нулями (если в какой-то местности не было зафиксировано потоков)
df_summary = df_summary.fillna(0)

# 4. СТРУКТУРНЫЙ АНАЛИЗ
# Рассчитываем долю влияния города на общий миграционный баланс региона (в %)
# Этот показатель — основа для проверки гипотез об урбанизации и субурбанизации
df_summary['Доля города в миграции'] = None
df_summary['Доля города в миграции'] = ((df_summary['Городская динамика'] / (df_summary['Городская динамика'] + df_summary['Сельская динамика'])) * 100).round(2)

# 5. КЛАССИФИКАЦИЯ РЕГИОНОВ
# Определяем тип демографической трансформации на основе векторов движения в городе и селе
conditions = [
    (df_summary['Городская динамика'] > 0) & (df_summary['Сельская динамика'] > 0),
    (df_summary['Городская динамика'] > 0) & (df_summary['Сельская динамика'] <= 0),
    (df_summary['Городская динамика'] <= 0) & (df_summary['Сельская динамика'] > 0),
    (df_summary['Городская динамика'] <= 0) & (df_summary['Сельская динамика'] <= 0)
]

labels = ['Общий рост', 'Урбанизация', 'Субурбанизация', 'Общий упадок']
# Применяем классификацию
df_summary['Тип трансформации'] = np.select(conditions, labels, default='Стагнация')

# 6. ЭКСПОРТ РЕЗУЛЬТАТОВ
# Сохраняем финальный датасет, который пойдет на вход в визуализацию и проверку гипотез
df_summary.to_csv('data/reports/structural_migration_shifts_2010_2023.csv', index=False, sep=';', encoding='utf-8-sig')